<a href="https://colab.research.google.com/github/eodenyire/LearnDataScienceWithEmmanuelOdenyire/blob/main/Phase_2_Data_Science_Core/Month_05_Data_Cleaning_and_Preprocessing/Week_4_Imbalanced_and_Augmentation/Week_5_Data_Augmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 5: Data Augmentation Techniques

## Learning Objectives
- Understand when and why to use data augmentation
- Apply feature-level augmentation for tabular data
- Use noise injection and synthetic data generation
- Understand image augmentation concepts

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_regression

rng = np.random.default_rng(42)
sns.set_theme(style='whitegrid')

## 1. Why Data Augmentation?

Data augmentation artificially expands training data to:
- Improve model generalisation
- Reduce overfitting on small datasets
- Balance class distributions
- Create more diverse training examples

## 2. Noise Injection for Tabular Data

In [ ]:
X, y = make_regression(n_samples=100, n_features=5, noise=10, random_state=42)
df = pd.DataFrame(X, columns=[f'feat_{i}' for i in range(5)])
df['target'] = y
print('Original shape:', df.shape)

# Gaussian noise augmentation
def augment_with_noise(df, noise_std=0.05, n_copies=1):
    augmented = [df.copy()]
    numeric_cols = df.select_dtypes(include=np.number).columns
    for _ in range(n_copies):
        noisy = df.copy()
        col_stds = df[numeric_cols].std()
        for col in numeric_cols:
            noisy[col] += rng.normal(0, noise_std * col_stds[col], len(df))
        augmented.append(noisy)
    return pd.concat(augmented, ignore_index=True)

df_aug = augment_with_noise(df, noise_std=0.1, n_copies=2)
print('Augmented shape:', df_aug.shape)

## 3. Feature Interpolation (Mixup)

In [ ]:
def mixup(df, alpha=0.2, n_samples=50):
    """Generate synthetic samples by interpolating pairs."""
    synthetic = []
    for _ in range(n_samples):
        idx1, idx2 = rng.integers(0, len(df), 2)
        lam = rng.beta(alpha, alpha)
        row = lam * df.iloc[idx1].values + (1 - lam) * df.iloc[idx2].values
        synthetic.append(row)
    return pd.DataFrame(synthetic, columns=df.columns)

df_mixup = mixup(df, n_samples=50)
print('Mixup synthetic data shape:', df_mixup.shape)
print('Mixup sample:\n', df_mixup.head(3).round(2))

## 4. Synthetic Data with Gaussian Copula

In [ ]:
# Simple Gaussian copula approach
from scipy.stats import norm

def generate_synthetic(df, n_samples=100):
    """Generate synthetic data preserving correlations."""
    numeric = df.select_dtypes(include=np.number)
    corr = numeric.corr().values
    L = np.linalg.cholesky(corr + 1e-6 * np.eye(len(corr)))
    z = rng.standard_normal((n_samples, len(numeric.columns)))
    z_corr = z @ L.T
    synthetic = pd.DataFrame(z_corr, columns=numeric.columns)
    # Scale to match original range
    for col in numeric.columns:
        synthetic[col] = synthetic[col] * numeric[col].std() + numeric[col].mean()
    return synthetic

df_syn = generate_synthetic(df.drop('target', axis=1), n_samples=100)
print('Synthetic data stats:\n', df_syn.describe().round(2))

## Exercise

1. Apply noise injection augmentation to a small dataset and evaluate how it affects model performance
2. Implement a Mixup augmentation function for a classification problem
3. Compare original vs augmented data distributions visually

In [ ]:
# Your code here


## Summary

This week you learned:
- ✓ Why and when to augment data
- ✓ Gaussian noise injection
- ✓ Mixup interpolation
- ✓ Synthetic data generation

## Next Month
In Month 6, we'll apply all these skills in Exploratory Data Analysis!